## 00-Split Pitt Data

In [1]:
# import required packages
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

### Define paths and constants

In [2]:
DATA_PATH = "../../../data/"
SPLIT_FOLDER = DATA_PATH + "splits/"
PITT_PATH_AD = DATA_PATH + "TalkBank downloads/Pitt/Dementia/Cookie/"
PITT_PATH_CN = DATA_PATH + "TalkBank downloads/Pitt/Control/Cookie/"

PARTICIPANT_INFO_PREFIX = "@ID"
AGE_THRESHOLD = 65
ID_INDEX = 0
AGE_INDEX = 2
MMSE_INDEX = -3

### Read data and keep earliest interview

In [3]:
def keep_first_interview(file_dict, lst):
    if lst[ID_INDEX] not in file_dict:
        file_dict[lst[ID_INDEX]] = lst
    else:
        if lst[AGE_INDEX] < file_dict[lst[ID_INDEX]][AGE_INDEX]:
            file_dict[lst[ID_INDEX]] = lst


def read_data(path, format, group, personal_info=False):
    files = sorted([item for item in os.listdir(path) if item.endswith(format)])
    file_dict = {}

    for fname in files:
        print("Processing files =", fname)
        with open(path + fname, encoding="utf-8") as f:
            lines = f.readlines()
            if personal_info:
                lst = [
                    line for line in lines if line.startswith(PARTICIPANT_INFO_PREFIX)
                ][ID_INDEX].split("|")
                lst = (
                    [fname.split("-")[ID_INDEX]]
                    + [group]
                    + lst[3:5]
                    + [lst[MMSE_INDEX]]
                )
                lst[AGE_INDEX] = int(lst[AGE_INDEX].split(";")[0])
                lst.append(lst[AGE_INDEX] < AGE_THRESHOLD)
                keep_first_interview(file_dict, lst)

    return list(file_dict.values())

In [4]:
pitt_id_train_data = {"ad": [], "cn": []}

In [5]:
format = "cha"
pitt_id_train_data["ad"] = read_data(SPLIT_FOLDER, format, "ad", True)
pitt_id_train_data["cn"] = read_data(PITT_PATH_CN, format, "cn", True)

Processing files = 002-0.cha
Processing files = 002-1.cha
Processing files = 002-2.cha
Processing files = 002-3.cha
Processing files = 006-2.cha
Processing files = 006-3.cha
Processing files = 006-4.cha
Processing files = 013-0.cha
Processing files = 013-2.cha
Processing files = 013-3.cha
Processing files = 013-4.cha
Processing files = 015-0.cha
Processing files = 015-1.cha
Processing files = 015-2.cha
Processing files = 015-3.cha
Processing files = 015-4.cha
Processing files = 017-4.cha
Processing files = 021-0.cha
Processing files = 021-1.cha
Processing files = 021-2.cha
Processing files = 021-3.cha
Processing files = 021-4.cha
Processing files = 022-0.cha
Processing files = 022-1.cha
Processing files = 022-2.cha
Processing files = 028-1.cha
Processing files = 028-4.cha
Processing files = 034-0.cha
Processing files = 034-1.cha
Processing files = 034-2.cha
Processing files = 034-3.cha
Processing files = 034-4.cha
Processing files = 042-1.cha
Processing files = 042-2.cha
Processing fil

In [6]:
comb_df = pd.DataFrame(
    pitt_id_train_data["ad"] + pitt_id_train_data["cn"],
    columns=["id", "target", "age_of_first_test", "sex", "mmse", "less_than_65"],
)

### Split data into train, dev, and test

In [7]:
comb_train_df, comb_test_df = train_test_split(
    comb_df,
    test_size=0.2,
    random_state=42,
    stratify=comb_df[["target", "less_than_65", "sex"]],
)

comb_train_df, comb_dev_df = train_test_split(
    comb_train_df,
    test_size=0.25,
    random_state=42,
    stratify=comb_train_df[["target", "less_than_65", "sex"]],
)

### Generate split CSV files

In [8]:
comb_train_df.to_csv(SPLIT_FOLDER + "pitt_train.csv", index=False)
comb_dev_df.to_csv(SPLIT_FOLDER + "pitt_dev.csv", index=False)
comb_test_df.to_csv(SPLIT_FOLDER + "pitt_test.csv", index=False)